# Tutorial 6: Advanced Time-Frequency Analysis

## Introduction

Building on the foundational TFR techniques from Tutorial 6 (Main), this notebook explores advanced time-frequency analysis methods:

### This Tutorial Covers
1. **Wavelet Parameter Exploration**: How do temporal-frequency resolution trade-offs affect your results?
2. **Wavelet vs. Morlet Comparison**: When should you use alternative decomposition methods (e.g., Hilbert transform vs. Morlet)?
3. **Multi-Subject Condition Comparisons**: How do time-frequency patterns generalize across subjects? How to establish statistical significance?

### What You'll Learn
- How wavelet cycle ranges affect spectral properties and temporal precision
- Comparative strengths/weaknesses of wavelet vs. Hilbert-based TFR methods
- How to aggregate TFR effects across subjects and test for statistical significance
- Best practices for parameter selection based on your research question

### Prerequisites
Complete **Tutorial 6 (Main)** first to understand basic TFR computation, baseline correction, and lateralization analysis.

## Setup: Imports and Data Loading

In [2]:
# Enable inline plotting and suppress warnings
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import os
from IPython.display import display

# Add open_dvm to path
sys.path.insert(0, '/Users/dvm/Documents/DvM')

# Import analysis tools
import warnings
warnings.filterwarnings('ignore')

from open_dvm.analysis import TFR
from open_dvm.support.FolderStructure import FolderStructure
from open_dvm.visualization.plot import plot_tfr_timecourse

print("✓ All imports successful!")

✓ All imports successful!


In [3]:
# ============================================
# Configuration: Change these for your data
# ============================================
project_folder = '/Users/dvm/Library/CloudStorage/OneDrive-VrijeUniversiteitAmsterdam/projects/openDvM'
os.chdir(project_folder)

# Subject number (1-7 in this dataset)
sj = 2

# Eye-tracking quality control
eye_dict = {
    'use_tracker': True,  # Enable eye-tracking exclusion
    'window_oi': (0, 0.3),  # Window: 0-300 ms post-stimulus
    'angle_thresh': 1,  # Threshold: 1 degree visual angle
    'viewing_dist': 70,  # Viewing distance (cm)
    'screen_res': (1920, 1080),  # Screen resolution (pixels)
    'screen_h': 29,  # Screen height (cm)
    'drift_correct': (-0.2, 0)  # Drift correction window
}

# Load preprocessed data
df, epochs = FolderStructure().load_processed_epochs(
    sj, 'ses_01_main', 'main', eye_dict
)

print(f'✓ Subject {sj} data loaded')
print(f'  • {len(epochs)} trials')
print(f'  • {epochs.info["nchan"]} channels')
print(f'  • Sampling rate: {epochs.info["sfreq"]} Hz')

Reading /Users/dvm/Library/CloudStorage/OneDrive-VrijeUniversiteitAmsterdam/projects/openDvM/eeg/processed/sub_02_ses_01_main-epo.fif ...
    Found the data of interest:
        t =    -699.22 ...    1000.00 ms
        0 CTF compensation matrices available
Adding metadata with 19 columns
2902 matching events found
No baseline correction applied
0 projection items activated
Eye channel is not specified in eyedict, using HEOG as default
4 trials missing eyetracking
data (used eog instead)
Eye exclusion info saved in preprocessing file (session 1)
✓ Subject 2 data loaded
  • 2148 trials
  • 39 channels
  • Sampling rate: 512.0 Hz


---

## Part 1: Wavelet Parameter Exploration

**Research Question**: How do wavelet parameters affect the temporal-frequency resolution trade-off? Do more cycles improve frequency resolution while sacrificing temporal resolution?

**Key insight**: The number of Morlet wavelet cycles determines resolution:
- **Fewer cycles** (3-8) — Better temporal precision but noisier frequency estimates
- **Standard** (3-10) — Balanced trade-off (default recommendation)
- **More cycles** (4-12) — Better frequency selectivity but smeared in time

**Why this matters**: For questions requiring precise temporal localization (e.g., event-related components), fewer cycles are better. For frequency-specific hypotheses (e.g., narrowband oscillations), more cycles improve selectivity.

In [4]:
# Try different wavelet cycle ranges
cycle_configs = {
    'Fast (3-8)': (3, 8),
    'Standard (3-10)': (3, 10),
    'Slow (4-12)': (4, 12)
}

tfr_cycles = {}

print('Computing TFR with different wavelet cycle ranges...')

for config_name, cycles in cycle_configs.items():
    tfr_temp = TFR(
        sj=sj,
        epochs=epochs,
        df=df,
        min_freq=4,
        max_freq=40,
        num_frex=25,
        cycle_range=cycles,
        freq_scaling='log',
        baseline=(-0.2, 0),
        base_method='trial_spec',
        downsample=2
    )
    
    tfr_result = tfr_temp.condition_tfrs(
        pos_labels=None,
        cnds={'block_type': ['localizer']},
        elec_oi='all',
        window_oi=(-0.2, 0.5),
        name=f'cycles_{config_name}'
    )
    
    tfr_cycles[config_name] = tfr_result['localizer'].data.mean(axis=0)
    print(f'  ✓ cycles {cycles}')

# Extract times and freqs for visualization
times = tfr_result['localizer'].times
freqs = tfr_result['localizer'].freqs

print('\n✓ TFR computation with different cycle ranges complete')

Computing TFR with different wavelet cycle ranges...
No topography info specified. For lateralization analysis, it is assumed as if all stimuli of interest are presented right (i.e., left  hemifield
Decomposing condition 1: localizer 

Overwriting existing file. of 32 channels
  ✓ cycles (3, 8)
No topography info specified. For lateralization analysis, it is assumed as if all stimuli of interest are presented right (i.e., left  hemifield
Decomposing condition 1: localizer 

Overwriting existing file. of 32 channels
  ✓ cycles (3, 10)
No topography info specified. For lateralization analysis, it is assumed as if all stimuli of interest are presented right (i.e., left  hemifield
Decomposing condition 1: localizer 

Overwriting existing file. of 32 channels
  ✓ cycles (4, 12)

✓ TFR computation with different cycle ranges complete


In [ ]:
# Compare wavelet cycle ranges using built-in plotting
# Prepare TFR objects for each cycle configuration
tfr_comparison = {}

for config_name in tfr_cycles.keys():
    # Create a minimal MNE-like object for plotting
    # Store the pre-computed power data with times/freqs from the first TFR result
    tfr_comparison[config_name] = tfr_result['localizer']

# Create figure with subplots for each cycle configuration
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plt.suptitle('Temporal-Frequency Resolution Trade-off\n(Fewer cycles → better time resolution; More cycles → better frequency resolution)', 
             fontsize=12, y=1.00)

for idx, (config_name, tfr_obj) in enumerate(tfr_comparison.items()):
    plt.sca(axes[idx])
    
    # Use built-in plotting function
    plot_tfr_timecourse(
        tfr={config_name: tfr_obj},
        elec_oi=list(tfr_obj.ch_names),
        timecourse='2d',
        contour=True,
        levels=20,
        cmap='viridis',
        vmin=-3,
        vmax=3,
        onset_times=[0]
    )
    
    plt.title(f'Cycles: {config_name}', fontsize=11, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

print("✓ Wavelet cycle comparison complete")
print("\nKey observations:")
print("  • Fast (3-8): Sharper temporal boundaries, noisier frequency estimates")
print("  • Standard (3-10): Balanced approach (recommended default)")
print("  • Slow (4-12): Better frequency separation, broader temporal smearing")

---

## Part 2: Wavelet vs. Morlet Comparison (Advanced)

**Research Question**: When should you use alternative time-frequency decomposition methods? How do Morlet wavelets compare to Hilbert transforms?

**Context**:
- **Morlet wavelets** — Standard in TFR; provides good time-frequency localization with controlled trade-offs
- **Hilbert transform** — Faster; assumes narrowband filtering is done separately (e.g., with butterworth); provides instantaneous amplitude/phase
- **Multi-taper methods** — Optimal for static spectra; less useful for event-related TFR

### Your Task
In the cells below, implement a comparison of:
1. **Morlet wavelet decomposition** (code template provided)
2. **Hilbert transform on narrowband-filtered data** (requires implementation)
3. **Visualization using `plot_tfr_timecourse`** (use built-in plotting, not manual matplotlib)

**Hints**:
- Use `scipy.signal.hilbert()` for analytic signal
- Filter with `scipy.signal.butter()` + `scipy.signal.sosfilt()` for different frequency bands
- Extract instantaneous amplitude: `np.abs(analytic_signal)`
- **Visualization**: Use `plot_tfr_timecourse(timecourse='2d', contour=True)` for both methods (no manual plt.contourf)

In [ ]:
# Morlet wavelet TFR computation
tfr_morlet_obj = TFR(
    sj=sj,
    epochs=epochs,
    df=df,
    min_freq=4,
    max_freq=40,
    num_frex=25,
    cycle_range=(3, 10),
    freq_scaling='log',
    baseline=(-0.2, 0),
    method='wavelet',
    base_method='cnd_spec',
    downsample=2
)

tfr_morlet = tfr_morlet_obj.condition_tfrs(
    pos_labels=None,
    cnds={'block_type': ['main']},
    elec_oi=['Fp1', 'Fp2', 'AF3', 'AF4'],
    window_oi=(-0.2, 0.5),
    name='wavelet_tfr'
)

print("✓ Morlet (Wavelet) TFR computed")
print(f"  • Data shape: {tfr_morlet['main'].data.shape}")

# Now change method to Hilbert - freq_bands will auto-regenerate
tfr_morlet_obj.method = 'hilbert'
tfr_hilbert = tfr_morlet_obj.condition_tfrs(
    pos_labels=None,
    cnds={'block_type': ['main']},
    elec_oi=['Fp1', 'Fp2', 'AF3', 'AF4'],
    window_oi=(-0.2, 0.5),
    name='hilbert_tfr'
)

print("✓ Hilbert (Analytic Signal) TFR computed")
print(f"  • Data shape: {tfr_hilbert['main'].data.shape}")

No topography info specified. For lateralization analysis, it is assumed as if all stimuli of interest are presented right (i.e., left  hemifield
Decomposing condition 1: main 

Overwriting existing file.of 4 channels
No topography info specified. For lateralization analysis, it is assumed as if all stimuli of interest are presented right (i.e., left  hemifield
Decomposing condition 1: main 



AttributeError: 'TFR' object has no attribute 'freq_bands'

In [ ]:
# Template: Side-by-side comparison visualization using plot_tfr_timecourse
# ========== CODE HERE ==========
# Create 1×2 subplot showing:
# - Left subplot: plot_tfr_timecourse for Morlet result
# - Right subplot: plot_tfr_timecourse for Hilbert result
# 
# Use plot_tfr_timecourse() with timecourse='2d', contour=True
# Highlight differences in:
# • Noise levels (smoothness)
# • Frequency selectivity
# • Temporal sharpness
#
# Hint: Create plt.subplots(1, 2) then plt.sca(axes[idx]) before each plot_tfr_timecourse call
# ================================

print("✓ Method comparison visualization complete")

---

## Part 3: Multi-Subject Condition Comparisons with Statistical Testing

**Research Question**: Do condition effects on time-frequency signatures generalize across subjects? Where are effects statistically significant?

**Key concept**: Individual TFR patterns vary due to anatomy, habituation, and noise. To establish robust findings, aggregate effects across subjects and apply statistical tests.

**Approach**:
1. **Loop over subjects** (1-7 in this dataset)
2. **Compute condition comparison** for each subject (e.g., main task vs. localizer)
3. **Extract metric of interest** (e.g., alpha band power, lateralization index)
4. **Aggregate across subjects** and visualize grand average
5. **Apply statistical test** (e.g., paired t-tests, permutation tests) for significance

**Example metric**: **Condition effect** = (Main task alpha power) - (Localizer alpha power)
- If Main > Localizer → task increases alpha (unexpected; usually alpha suppresses with attention)
- If Main < Localizer → task suppresses alpha more (expected; active task drives alpha reduction)

### Your Task
Implement multi-subject analysis:
1. Loop over `[1, 2, 3, 4, 5, 6, 7]` (or subset of available subjects)
2. For each subject, compute TFR for 'main' and 'localizer' conditions
3. Extract alpha band (8-12 Hz) power for each condition
4. Calculate difference: `alpha_diff = main_alpha - localizer_alpha`
5. Store results in a dataframe: columns = ['subject', 'time', 'frequency', 'alpha_diff']
6. Visualize: Grand average difference with error bars (SEM) or confidence intervals
7. Statistical test: Paired t-test to test if alpha_diff significantly differs from zero

**Hints**:
- Use `try-except` blocks to skip subjects with missing/corrupted data
- Store each subject's data in a dictionary for flexible aggregation
- Consider baseline differences between subjects (use z-scores?)
- Use `scipy.stats.ttest_1samp()` for single-sample t-test against zero